In [181]:
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

from diffusion import Diffusion
from dataset import DiffusionDataset, get_dataloader

In [217]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.datasets import fetch_openml
from sklearn.preprocessing import StandardScaler
from diffusion import Diffusion

class DiffusionDataset(Dataset):
    def __init__(self, dataset_name:str, x_normalise:bool=False, y_normalise:bool=False, seed:int=None):
        """
        Dataset class for diffusion Gaussian processes.

        Args:
            dataset_name (str): Name of the dataset to load.
            x_normalise (bool): Whether to normalise x.
            y_normalise (bool): Whether to normalise y.
            seed (int): Random seed.
            x (torch.Tensor/np.ndarray): Custom input features of shape (n_samples, n_features).
            y (torch.Tensor/np.ndarray): Custom target values of shape (n_samples, 1).

        Returns:
            x (torch.Tensor): Input features of shape (n_samples, n_features).
            y (torch.Tensor): Target values of shape (n_samples, 1).
        """
        # Initialisations
        self.dataset_name = dataset_name
        self.x_normalise = x_normalise
        self.y_normalise = y_normalise
        self.x_scaler = None
        self.y_scaler = None
        self.seed = seed 
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        if self.seed is not None: # seed for generating sin_1d dataset
            np.random.seed(self.seed)

        # Load raw datasets
        if dataset_name == 'sin_1d':
            x, mean, std = self.sin_1d()
            self.x = x
            self.y = (mean + std * np.random.randn(x.shape[0]))
        else: # UCI datasets
            self.x, self.y = fetch_openml(dataset_name, version=1, return_X_y=True, as_frame=False)
        
        # Reshape x and y to 2D if they are 1D
        if self.x.ndim == 1:
            print(f"Reshaping x from 1D to 2D, shape: {self.x.shape} -> {self.x.shape[0],1}")
            self.x = self.x.reshape(-1,1)
        if self.y.ndim == 1:
            print(f"Reshaping y from 1D to 2D, shape: {self.y.shape} -> {self.y.shape[0],1}")
            self.y = self.y.reshape(-1,1)
        assert self.x.ndim == 2 and self.y.ndim == 2, f"x and y must be 2D arrays, but got {self.x.ndim} and {self.y.ndim}"

        # Normalisation to mean 0 and std 1
        if self.x_normalise:
            self.x_scaler = StandardScaler()
            self.x = self.x_scaler.fit_transform(self.x)
        if self.y_normalise:
            self.y_scaler = StandardScaler()
            self.y = self.y_scaler.fit_transform(self.y)

        # Convert to torch tensors with float32 dtype
        self.x = torch.tensor(self.x, dtype=torch.float32, device=self.device).detach().clone()
        self.y = torch.tensor(self.y, dtype=torch.float32, device=self.device).detach().clone()

    def __len__(self) -> int:
        return len(self.x)
    
    def __getitem__(self, idx:int) -> dict:
        return {'x': self.x[idx], 'y': self.y[idx]}

    @staticmethod
    def sin_1d(n_samples:int=5000) -> tuple[np.ndarray, np.ndarray]: 
        # True underlying distribution
        def f(x):
            return 1 + np.sin(2 * np.pi * x)
        def std(x):
            return 0.5 + 0.4 * np.cos(6 * np.pi * x)
        
        # Generate random x values
        x = np.random.rand(n_samples)
        return x, f(x), std(x) # shape (n_samples,)
    
class AugmentedDataset(Dataset):
    def __init__(self, dataset:DiffusionDataset, T:int, target:str='noise'):
        self.T = T
        self.target = target
        self.x = torch.empty((len(dataset.x)*(T-1), dataset.x.shape[1]+2))
        self.y = torch.empty((len(dataset.x)*(T-1), 1))
        diffusion = Diffusion(T=T)

        # Augment the dataset
        print(f"Augmenting dataset with {T} time steps, target type: {target}")
        with tqdm(total=len(dataset.x)) as pbar:
            for i in range(len(dataset.x)):
                for t in range(1, T):
                    noise = torch.randn_like(dataset.y[i])
                    y_t = diffusion.sample_q(dataset.y[i], t, noise)
                    self.x[i*(T-1)+t-1] = torch.cat([dataset.x[i], torch.tensor(t).unsqueeze(0)/T, y_t], dim=0)

                    # Check targe type
                    if target == 'noise':
                        self.y[i*(T-1)+t-1] = noise
                    elif target == 'y_t':
                        self.y[i*(T-1)+t-1] = dataset.y[i]
                pbar.update(1)

    def __len__(self) -> int:
        return len(self.x)
    
    def __getitem__(self, idx:int) -> dict:
        return {'x': self.x[idx], 'y': self.y[idx]}


def get_dataloader(dataset:DiffusionDataset, batch_size:int, train_ratio:float=0.8, val_ratio:float=0.1, shuffle:bool=True, seed:int=None) -> tuple[DataLoader, DataLoader, DataLoader]:
    """
    Get dataloaders for training, validation and test sets.

    Args:
        dataset (DiffusionDataset): Dataset to split.
        batch_size (int): Batch size.
        train_ratio (float): Ratio of training set.
        val_ratio (float): Ratio of validation set.
        shuffle (bool): Whether to shuffle the training dataset.
        seed (int): Random seed for splitting the dataset.

    Returns:
        train_loader (DataLoader): Dataloader for training set.
        val_loader (DataLoader): Dataloader for validation set.
        test_loader (DataLoader): Dataloader for test set.
    """
    # Initialisations
    if seed is not None: # seed for splitting the dataset
        torch.manual_seed(seed)
    
    # Split dataset
    n_samples = len(dataset)
    n_train = int(train_ratio * n_samples)
    n_val = int(val_ratio * n_samples)
    n_test = n_samples - n_train - n_val
    train_dataset, val_dataset, test_dataset = torch.utils.data.random_split(dataset,[n_train, n_val, n_test]) # splits into training and validation datasets of desired ratio
    
    # Create dataloaders: divide each dataset into batches
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=shuffle, drop_last=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, drop_last=False)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, drop_last=False)

    return train_loader, val_loader, test_loader

In [212]:
# Parameters
datasest_param = {
    'dataset_name': 'sin_1d',
    'x_normalise': True,
    'y_normalise': True,
    'seed': 42,
    'batch_size': 32,
    'train_ratio': 0.8,
    'val_ratio': 0.1,
    'shuffle': True,
}
diffusion_param = {
    'T': 1000,
}

# Get dataset
dataset = DiffusionDataset(dataset_name=datasest_param['dataset_name'], x_normalise=datasest_param['x_normalise'], y_normalise=datasest_param['y_normalise'], seed=datasest_param['seed'])
if datasest_param['x_normalise']:
    x_scaler = dataset.x_scaler
if datasest_param['y_normalise']:
    y_scaler = dataset.y_scaler
train_loader, val_loader, test_loader = get_dataloader(dataset, batch_size=datasest_param['batch_size'], train_ratio=datasest_param['train_ratio'], val_ratio=datasest_param['val_ratio'], shuffle=datasest_param['shuffle'], seed=datasest_param['seed'])

Reshaping x from 1D to 2D, shape: (5000,) -> (5000, 1)
Reshaping y from 1D to 2D, shape: (5000,) -> (5000, 1)


In [213]:
aug_dataset = augment_dataset(dataset, T=10, target='noise')

Augmenting dataset with 10 time steps, target type: noise


100%|██████████| 5000/5000 [00:00<00:00, 5489.97it/s]
/var/folders/sq/_ygl92r55ks2__x_h38qg7h00000gn/T/ipykernel_29329/2413643501.py:65: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.x = torch.tensor(self.x, dtype=torch.float32, device=self.device).detach().clone()
/var/folders/sq/_ygl92r55ks2__x_h38qg7h00000gn/T/ipykernel_29329/2413643501.py:66: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.y = torch.tensor(self.y, dtype=torch.float32, device=self.device).detach().clone()
